# Complete SDS1 Analysis: Exploring Factor Effects

This tutorial demonstrates how to analyze a fully replicated factorial design (SDS 1) using the `by` parameter to explore different views of the same data.

## What You'll Learn

1. Load and formulate a two-factor study
2. Use the `by` parameter to aggregate Xbar/S charts at different levels
3. Stratify IMR charts by factor combinations
4. Understand lane boundaries in collapsed charts
5. Chart residuals using the `value` parameter

## Setup

In [ ]:
import pandas as pd
from processbehavior import ProcessBehavior

## 1. Load and Formulate

We'll use the SDS1 validation dataset which has:
- **factor 1**: 3 levels (F1_1, F1_2, F1_3)
- **factor 2**: 2 levels (F2_1, F2_2)
- **time**: 8 periods
- Multiple replicates per cell

In [ ]:
# Load the SDS1 validation data
df = pd.read_csv('../validation/sds1_data.csv')

print(f"Dataset: {df.shape[0]} observations")
print(f"Factor 1 levels: {df['factor 1'].unique().tolist()}")
print(f"Factor 2 levels: {df['factor 2'].unique().tolist()}")
print(f"Time periods: {sorted(df['time'].unique())}")
df.head()

In [ ]:
# Formulate the study with both factors
pb = ProcessBehavior(df)
study = pb.formulate(
    response='y',
    factors=['factor 1', 'factor 2'],
    time='time'
)

print(f"SDS: {study.sds} ({study.sds_name})")
print(f"Valid charts: {study.valid_charts}")
print(f"Available residuals: {study.available_residuals}")

## 2. Xbar Charts - Factor Aggregation

The `by` parameter controls how data points are aggregated on Xbar charts:
- **Default (all factors)**: One point per factor combination (6 points)
- **Single factor**: Aggregate across the other factor
- **Empty list**: Collapse to grand mean (1 point)

### 2.1 Xbar by All Factors (Default)

In [ ]:
# Default: aggregate by all factors
result = study.execute(chart='Xbar')

print("Xbar chart data (one point per factor combination):")
result.get_chart('Xbar')

In [ ]:
result.plot(chart='Xbar', show_stats=True)

### 2.2 Xbar by Factor 1 Only

In [ ]:
# Aggregate by factor 1 only (3 points, one per F1 level)
result_f1 = study.execute(chart='Xbar', by=['factor 1'])

print("Xbar aggregated by factor 1:")
result_f1.get_chart('Xbar')

In [ ]:
result_f1.plot(chart='Xbar', show_stats=True)

### 2.3 Xbar by Factor 2 Only

In [ ]:
# Aggregate by factor 2 only (2 points, one per F2 level)
result_f2 = study.execute(chart='Xbar', by=['factor 2'])

print("Xbar aggregated by factor 2:")
result_f2.get_chart('Xbar')

In [ ]:
result_f2.plot(chart='Xbar', show_stats=True)

### 2.4 Xbar Collapsed (Grand Mean)

In [ ]:
# Collapse all factors (single point - grand mean)
result_all = study.execute(chart='Xbar', by=[])

print("Xbar collapsed to grand mean:")
result_all.get_chart('Xbar')

## 3. S Charts - Variation Analysis

S charts follow the same `by` parameter logic as Xbar charts.

In [ ]:
# S chart by all factors (default)
result_s = study.execute(chart='S')

print("S chart (within-group standard deviation):")
result_s.get_chart('S')

In [ ]:
result_s.plot(chart='S', show_stats=True)

In [ ]:
# S chart by factor 1 only
result_s_f1 = study.execute(chart='S', by=['factor 1'])

print("S chart aggregated by factor 1:")
result_s_f1.get_chart('S')

In [ ]:
# S chart by factor 2 only
result_s_f2 = study.execute(chart='S', by=['factor 2'])

print("S chart aggregated by factor 2:")
result_s_f2.get_chart('S')

## 4. IMR Charts - Stratified Analysis

IMR charts with factors **require** an explicit `by` parameter. The `by` parameter controls stratification:
- **Both factors**: Separate chart for each factor combination
- **Single factor**: Charts per level with lane boundaries showing the other factor
- **Empty list**: Single chart with lane boundaries for all factor transitions

### 4.1 IMR by Both Factors (6 Faceted Charts)

In [ ]:
# IMR stratified by both factors
result_imr = study.execute(chart='Imr', by=['factor 1', 'factor 2'])

print(f"Strata: {result_imr.charts['Imr']['strata']}")
print(f"Each stratum has its own IMR chart")

In [ ]:
result_imr.plot(chart='Imr', show_zones=True)

### 4.2 IMR by Factor 1 Only (3 Charts with Lane Boundaries)

When stratifying by one factor, the collapsed factor creates multiple observations at each time point. **Lane boundaries** show where the collapsed factor changes.

In [ ]:
# IMR stratified by factor 1 only
result_imr_f1 = study.execute(chart='Imr', by=['factor 1'])

print(f"Strata: {result_imr_f1.charts['Imr']['strata']}")
print("\nLane boundaries show where factor 2 changes within each chart")

In [ ]:
result_imr_f1.plot(chart='Imr', show_zones=True)

### 4.3 IMR by Factor 2 Only (2 Charts with Lane Boundaries)

In [ ]:
# IMR stratified by factor 2 only
result_imr_f2 = study.execute(chart='Imr', by=['factor 2'])

print(f"Strata: {result_imr_f2.charts['Imr']['strata']}")
print("\nLane boundaries show where factor 1 changes within each chart")

In [ ]:
result_imr_f2.plot(chart='Imr', show_zones=True)

### 4.4 Single IMR (Collapsed, with Lane Boundaries)

In [ ]:
# Single IMR chart with all factors collapsed
result_imr_all = study.execute(chart='Imr', by=[])

print("Single IMR chart with all data")
print("Lane boundaries show transitions between factor combinations")

In [ ]:
result_imr_all.plot(chart='Imr', show_zones=True)

## 5. Residual Charts

Use the `value` parameter to chart VAS residuals instead of the response variable.

### 5.1 R5 (Factor Effects) on Xbar

In [ ]:
# R5 residuals show factor effects
result_r5 = study.execute(chart='Xbar', value='R5')

print("R5 Xbar chart (factor effects):")
result_r5.get_chart('Xbar')

In [ ]:
result_r5.plot(chart='Xbar', show_stats=True)

### 5.2 Recentered Residuals

Use `recentered=True` to center residuals around zero.

In [ ]:
# Recentered R5 residuals
result_r5_rc = study.execute(chart='Xbar', value='R5', recentered=True)

print("Recentered R5 Xbar chart:")
result_r5_rc.get_chart('Xbar')

In [ ]:
result_r5_rc.plot(chart='Xbar', show_stats=True)

### 5.3 R5 on S Chart

In [ ]:
# R5 on S chart
result_r5_s = study.execute(chart='S', value='R5')
result_r5_s.plot(chart='S', show_stats=True)

## Summary

### When to Use Each `by` Configuration

| Configuration | Use Case |
|--------------|----------|
| `by=None` (default) | Compare all factor combinations |
| `by=['factor 1']` | Focus on one factor, aggregate the other |
| `by=[]` | Overall process view, collapsed factors |
| `by=['factor 1', 'factor 2']` | Individual charts per combination |

### Key Concepts

1. **Views, Not Recomputation**: The `by` parameter creates views over the same underlying data. Residuals never change.

2. **Lane Boundaries**: When IMR charts collapse factors, vertical lane boundaries show where factor transitions occur.

3. **Residuals via `value`**: Use `value='R5'` to chart residuals instead of response.

4. **Recentering**: Use `recentered=True` to center residuals around zero.